[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/04_Opset_and_Metadata/Opset_and_Metadata_Apply.ipynb)

# 1.4 Opset and Metadata — Hands-On Practice

## Objective

Learn how to control **opset versions** and attach rich **metadata** to ONNX models.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | Setup | Dependencies |
| 2 | Exercise 1: Opset Versions | Build with different opset levels |
| 3 | Exercise 2: Query `since_version` | Find when operators were introduced |
| 4 | Exercise 3: Multi-Domain Imports | Standard + custom opset |
| 5 | Exercise 4: Model Metadata | Annotate models with producer info |
| 6 | Exercise 5: Custom `metadata_props` | Key-value metadata for ML-ops |
| 7 | Exercise 6: Version Conversion | Upgrade/downgrade opset |
| 8 | Exercise 7: Metadata Audit | Build a compliance checker |
| 9 | Exercise 8: Opset Compatibility Matrix | Test across versions |
| 10 | Challenge: Model Registry Card | Rich metadata for tracking |

<a id='section-1'></a>
## Section 1: Setup

In [ ]:
# !pip install onnx onnxruntime numpy

import numpy as np
import os
import time
from datetime import datetime

import onnx
from onnx import TensorProto, load, save, version_converter
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx.numpy_helper import from_array, to_array
from onnx.defs import get_schema, onnx_opset_version
import onnxruntime as ort

print(f'ONNX version:    {onnx.__version__}')
print(f'Default opset:   {onnx_opset_version()}')
print(f'ORT version:     {ort.__version__}')

<a id='section-2'></a>
## Section 2: Exercise 1 — Build Models with Different Opset Levels

### What is an Opset?

The **opset** (operator set) version determines which operators and which
*semantics* of each operator are available.  Think of it like an API version.

- Opset 7: Early, limited set
- Opset 13: Added `Squeeze`/`Unsqueeze` taking axes as input
- Opset 17–21: Latest operators

We build the same $Y = X + B$ graph at three different opset levels.

In [ ]:
def build_add_model(opset_version):
    """Build Y = X + B at a given opset."""
    X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
    B = make_tensor_value_info('B', TensorProto.FLOAT, [4])
    Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])
    graph = make_graph(
        [make_node('Add', ['X', 'B'], ['Y'])],
        f'add_opset{opset_version}', [X, B], [Y])
    model = make_model(graph, opset_imports=[make_opsetid('', opset_version)])
    check_model(model)
    return model

for v in [9, 13, 17]:
    m = build_add_model(v)
    opset_actual = m.opset_import[0].version
    ir_ver = m.ir_version
    print(f'  opset={opset_actual:3d}  ir_version={ir_ver}  '
          f'size={len(m.SerializeToString()):,} bytes')

    # Verify all produce the same result
    sess = ort.InferenceSession(
        m.SerializeToString(), providers=['CPUExecutionProvider'])
    x = np.array([[1, 2, 3, 4]], dtype=np.float32)
    b = np.array([10, 20, 30, 40], dtype=np.float32)
    result = sess.run(None, {'X': x, 'B': b})[0]
    assert np.allclose(result, x + b)

print('All opset versions produce correct results!')

<a id='section-3'></a>
## Section 3: Exercise 2 — Query `since_version`

Each operator was introduced at a specific opset version. The `since_version`
field tells you when the *current signature* was defined.

In [ ]:
operators_to_check = [
    'Add', 'MatMul', 'Conv', 'Relu', 'BatchNormalization',
    'Softmax', 'Reshape', 'Squeeze', 'Unsqueeze', 'Gemm',
    'LayerNormalization', 'ReduceMean', 'Flatten', 'Concat',
    'Transpose', 'Gather', 'Slice', 'Pad',
]

print(f'{"Operator":25s} {"Since":>6s}  {"Stable Since":>13s}')
print('-' * 50)

for op in operators_to_check:
    try:
        schema = get_schema(op, domain='')
        since = schema.since_version
        # Find earliest version that supports this op
        versions = []
        for v in range(1, onnx_opset_version() + 1):
            try:
                s = get_schema(op, v, domain='')
                versions.append(v)
            except Exception:
                pass
        first = min(versions) if versions else '?'
        print(f'{op:25s} {since:>6d}  {str(first):>13s}')
    except Exception as e:
        print(f'{op:25s}  ERROR: {e}')

<a id='section-4'></a>
## Section 4: Exercise 3 — Multi-Domain Opset Imports

A model can import operators from **multiple domains**:
- `''` (empty) = standard ONNX domain
- `'ai.onnx.ml'` = ML-specific operators (e.g., TreeEnsembleClassifier)
- Custom domains for your own operators

The `opset_import` field is a *list* of `(domain, version)` pairs.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])

graph = make_graph(
    [make_node('Relu', ['X'], ['Y'])],
    'multi_domain', [X], [Y])

model_md = make_model(
    graph,
    opset_imports=[
        make_opsetid('', 17),            # standard ONNX
        make_opsetid('ai.onnx.ml', 3),   # ML operators
        make_opsetid('my.custom', 1),     # custom domain
    ])

print('Opset imports:')
for oi in model_md.opset_import:
    domain = oi.domain if oi.domain else '(default/ai.onnx)'
    print(f'  domain="{domain}"  version={oi.version}')

# Verify the model runs (only standard ops used)
check_model(model_md)
sess = ort.InferenceSession(
    model_md.SerializeToString(), providers=['CPUExecutionProvider'])
x = np.array([[-1, 2, -3, 4]], dtype=np.float32)
result = sess.run(None, {'X': x})[0]
assert np.allclose(result, np.maximum(0, x))
print('\nInference OK despite extra domain imports!')

<a id='section-5'></a>
## Section 5: Exercise 4 — Model Metadata Fields

ONNX `ModelProto` has built-in metadata fields:

| Field | Purpose |
|-------|--------|
| `producer_name` | Tool that created the model |
| `producer_version` | Tool version |
| `domain` | Model domain (e.g., `ai.vision`) |
| `model_version` | User-defined model version number |
| `doc_string` | Free-text description |

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])
graph = make_graph(
    [make_node('Relu', ['X'], ['Y'])], 'annotated', [X], [Y])

model_meta = make_model(graph, opset_imports=[make_opsetid('', 17)])

# Set metadata fields
model_meta.producer_name = 'ONNX Tutorial'
model_meta.producer_version = '2.0.0'
model_meta.domain = 'ai.tutorial.demo'
model_meta.model_version = 3
model_meta.doc_string = (
    'Demo model computing Relu(X). '
    'Used for teaching metadata in the ONNX Tutorial series.')

check_model(model_meta)

# Display all fields
fields = [
    ('ir_version', model_meta.ir_version),
    ('producer_name', model_meta.producer_name),
    ('producer_version', model_meta.producer_version),
    ('domain', model_meta.domain),
    ('model_version', model_meta.model_version),
    ('doc_string', model_meta.doc_string[:80] + '...'),
    ('graph.name', model_meta.graph.name),
]

max_key = max(len(k) for k, _ in fields)
for k, v in fields:
    print(f'  {k:{max_key}s} : {v}')

<a id='section-6'></a>
## Section 6: Exercise 5 — Custom `metadata_props` (ML-Ops Tags)

Beyond the built-in fields, `metadata_props` is a list of arbitrary
key-value pairs — perfect for ML-ops tracking, experiment IDs, dataset hashes, etc.

```
metadata_props:
  experiment_id = "exp-2024-12-15-v3"
  dataset       = "cifar10_train_v2"
  accuracy      = "0.923"
  commit_hash   = "a1b2c3d4"
```

In [ ]:
from onnx.helper import make_model, make_graph, make_node, make_tensor_value_info, make_opsetid
from onnx import StringStringEntryProto

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 10])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 10])
g = make_graph([make_node('Relu', ['X'], ['Y'])], 'tagged_model', [X], [Y])
model_tagged = make_model(g, opset_imports=[make_opsetid('', 17)])
model_tagged.producer_name = 'ONNX Tutorial'

# Add custom metadata
tags = {
    'experiment_id': 'exp-2024-12-15-v3',
    'dataset': 'cifar10_train_v2',
    'accuracy': '0.923',
    'f1_score': '0.917',
    'commit_hash': 'a1b2c3d4',
    'author': 'ml-team',
    'created_at': datetime.now().isoformat(),
    'framework': 'pytorch-2.1',
    'license': 'Apache-2.0',
}

for key, value in tags.items():
    entry = model_tagged.metadata_props.add()
    entry.key = key
    entry.value = value

check_model(model_tagged)

# Retrieve and display
print('Custom metadata_props:')
print('-' * 50)
for prop in model_tagged.metadata_props:
    print(f'  {prop.key:20s} = {prop.value}')

# Save, reload, verify metadata survives serialization
save(model_tagged, 'tagged_model.onnx')
reloaded = load('tagged_model.onnx')
reloaded_tags = {p.key: p.value for p in reloaded.metadata_props}

print(f'\nMetadata survives save/load: {reloaded_tags["experiment_id"] == "exp-2024-12-15-v3"}')
print(f'All keys present: {set(tags.keys()) == set(reloaded_tags.keys())}')
os.remove('tagged_model.onnx')

<a id='section-7'></a>
## Section 7: Exercise 6 — Opset Version Conversion

ONNX provides `version_converter.convert_version()` to upgrade or downgrade
a model's opset version. This is useful when deploying to runtimes that
only support specific opset ranges.

In [ ]:
# Build at opset 11
base = build_add_model(11)
print(f'Original opset: {base.opset_import[0].version}')

# Upgrade to opset 17
upgraded = version_converter.convert_version(base, 17)
print(f'Upgraded opset:  {upgraded.opset_import[0].version}')

# Both should produce the same result
x = np.array([[1, 2, 3, 4]], dtype=np.float32)
b = np.array([10, 20, 30, 40], dtype=np.float32)

for label, mdl in [('original', base), ('upgraded', upgraded)]:
    sess = ort.InferenceSession(
        mdl.SerializeToString(), providers=['CPUExecutionProvider'])
    res = sess.run(None, {'X': x, 'B': b})[0]
    print(f'  {label:10s} opset={mdl.opset_import[0].version:2d}  '
          f'result={res.tolist()}')

print('Outputs match after opset conversion!')

<a id='section-8'></a>
## Section 8: Exercise 7 — Metadata Audit Tool

Build a compliance checker that enforces required metadata fields.
This is useful in production ML pipelines where every model file
must carry experiment tracking info.

In [ ]:
def audit_model_metadata(model, required_fields=None, required_props=None):
    """Check that a model meets metadata requirements."""
    if required_fields is None:
        required_fields = ['producer_name', 'doc_string']
    if required_props is None:
        required_props = ['author', 'license']

    issues = []

    # Check built-in fields
    for field in required_fields:
        val = getattr(model, field, '')
        if not val:
            issues.append(f'Missing required field: {field}')

    # Check custom metadata_props
    existing_keys = {p.key for p in model.metadata_props}
    for key in required_props:
        if key not in existing_keys:
            issues.append(f'Missing required metadata_prop: {key}')

    # Check opset
    if model.opset_import[0].version < 11:
        issues.append(
            f'Opset {model.opset_import[0].version} too old (min 11)')

    passed = len(issues) == 0
    status = 'PASS' if passed else 'FAIL'
    print(f'Audit result: [{status}]')
    if not passed:
        for issue in issues:
            print(f'  ! {issue}')
    return passed, issues


print('=== Model with metadata ===')
ok, _ = audit_model_metadata(
    model_meta,
    required_fields=['producer_name', 'doc_string'],
    required_props=[])
assert ok

print('\n=== Bare model (should fail) ===')
bare = build_add_model(13)
ok, issues = audit_model_metadata(bare)
assert not ok
print(f'  Found {len(issues)} issues')

print('\n=== Model with tags ===')
ok, _ = audit_model_metadata(
    model_tagged,
    required_props=['author', 'license', 'experiment_id'])
assert ok

<a id='section-9'></a>
## Section 9: Exercise 8 — Opset Compatibility Matrix

Test which opset versions can successfully run a given model and
build a compatibility matrix.

In [ ]:
ops_to_test = ['Add', 'MatMul', 'Relu', 'Sigmoid', 'Softmax']
opset_range = range(7, min(onnx_opset_version() + 1, 22))

print(f'{"Op":15s}', end='')
for v in opset_range:
    print(f'{v:>4d}', end='')
print()
print('-' * (15 + 4 * len(opset_range)))

for op in ops_to_test:
    print(f'{op:15s}', end='')
    for v in opset_range:
        try:
            # Two-input ops
            if op in ('Add', 'MatMul'):
                X1 = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
                X2 = make_tensor_value_info('W', TensorProto.FLOAT, [4, 4])
                Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])
                g = make_graph(
                    [make_node(op, ['X', 'W'], ['Y'])],
                    'test', [X1, X2], [Y])
            else:
                X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
                Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])
                if op == 'Softmax':
                    g = make_graph(
                        [make_node(op, ['X'], ['Y'], axis=-1)],
                        'test', [X], [Y])
                else:
                    g = make_graph(
                        [make_node(op, ['X'], ['Y'])],
                        'test', [X], [Y])
            m = make_model(g, opset_imports=[make_opsetid('', v)])
            check_model(m)
            print('  OK', end='')
        except Exception:
            print('   -', end='')
    print()

<a id='section-10'></a>
## Section 10: Challenge — Model Registry Card Generator

Combine everything into a function that generates a rich "model card"
with full metadata, suitable for a model registry or deployment dashboard.

```
┌──────────────────────────────────────────────────┐
│              MODEL REGISTRY CARD                 │
├──────────────────────────────────────────────────┤
│ Name:      cifar10_cnn_v3                        │
│ Author:    ml-team                               │
│ Opset:     17                                    │
│ Accuracy:  92.3%                                 │
│ License:   Apache-2.0                            │
│ Created:   2024-12-15T10:30:00                   │
└──────────────────────────────────────────────────┘
```

In [ ]:
def create_registered_model(graph, *, name, author, version, accuracy,
                            dataset, license_str='Apache-2.0',
                            opset=17, extra_tags=None):
    """Build an ONNX model with full registry metadata."""
    model = make_model(graph, opset_imports=[make_opsetid('', opset)])
    model.producer_name = 'ONNX Tutorial Registry'
    model.producer_version = '1.0.0'
    model.domain = 'ai.tutorial.registry'
    model.model_version = version
    model.doc_string = f'Registered model: {name}'

    required_tags = {
        'model_name': name,
        'author': author,
        'model_version': str(version),
        'accuracy': str(accuracy),
        'dataset': dataset,
        'license': license_str,
        'created_at': datetime.now().isoformat(),
        'opset': str(opset),
    }
    if extra_tags:
        required_tags.update(extra_tags)

    for k, v in required_tags.items():
        entry = model.metadata_props.add()
        entry.key = k
        entry.value = v

    check_model(model)
    return model


def print_model_card(model):
    """Pretty-print the model registry card."""
    tags = {p.key: p.value for p in model.metadata_props}
    w = 54
    print('\n' + '+' + '-' * w + '+')
    print('|' + 'MODEL REGISTRY CARD'.center(w) + '|')
    print('+' + '-' * w + '+')
    card_fields = [
        ('Name', tags.get('model_name', 'N/A')),
        ('Author', tags.get('author', 'N/A')),
        ('Version', tags.get('model_version', 'N/A')),
        ('Opset', str(model.opset_import[0].version)),
        ('IR Version', str(model.ir_version)),
        ('Accuracy', tags.get('accuracy', 'N/A')),
        ('Dataset', tags.get('dataset', 'N/A')),
        ('License', tags.get('license', 'N/A')),
        ('Created', tags.get('created_at', 'N/A')[:19]),
        ('Producer', model.producer_name),
        ('Nodes', str(len(model.graph.node))),
        ('Inputs', ', '.join(i.name for i in model.graph.input)),
        ('Outputs', ', '.join(o.name for o in model.graph.output)),
    ]
    for label, val in card_fields:
        line = f'  {label:14s}: {val}'
        print(f'| {line:{w-1}s}|')
    print('+' + '-' * w + '+')


# Build a demo model and register it
W_init = from_array(np.random.randn(4, 2).astype(np.float32), 'W')
b_init = from_array(np.zeros(2, dtype=np.float32), 'b')
X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 2])
g = make_graph(
    [make_node('MatMul', ['X', 'W'], ['XW']),
     make_node('Add', ['XW', 'b'], ['Y_pre']),
     make_node('Relu', ['Y_pre'], ['Y'])],
    'classifier', [X], [Y], [W_init, b_init])

registered = create_registered_model(
    g, name='cifar10_cnn_v3', author='ml-team',
    version=3, accuracy=0.923, dataset='cifar10_train_v2',
    extra_tags={'commit_hash': 'a1b2c3d4', 'framework': 'pytorch-2.1'})

print_model_card(registered)

# Audit passes
ok, _ = audit_model_metadata(
    registered,
    required_fields=['producer_name', 'doc_string'],
    required_props=['author', 'license', 'accuracy'])
assert ok, 'Registry model should pass audit'

---

## Summary

| Exercise | Topic | Key API |
|----------|-------|---------|
| 1 | Opset versions | `make_opsetid('', version)` |
| 2 | `since_version` | `onnx.defs.get_schema()` |
| 3 | Multi-domain imports | List of `make_opsetid` |
| 4 | Built-in metadata | `producer_name`, `model_version`, etc. |
| 5 | `metadata_props` | Key-value pairs for ML-ops tracking |
| 6 | Version conversion | `version_converter.convert_version()` |
| 7 | Audit tool | Enforce metadata compliance |
| 8 | Compatibility matrix | Test ops across opset range |
| Challenge | Registry card | Production-ready model tagging |

**Next:** [Subgraphs, Tests, and Loops](../05_Subgraphs_Tests_Loops/)